### RAG Pipelines - Data Ingestion To Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path


C:\Users\HP\AppData\Local\Temp\ipykernel_6928\1867757213.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
e:\generative ai project\RAG Tutorial\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
### Read all the PDF's inside the directory

def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files Recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # add source information to metadata

            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            all_documents.extend(documents)
            print(f" ✔ loaded {len(documents)} pages")
        except Exception as e:
            print(f"  ❌ Error: {e}")
    print(f"\nTotal Documents Loaded: {len(all_documents)}")
    return all_documents
# Process all PDF's In the data directory

all_pdf_documents = process_all_pdfs("../data")

Found 2 PDF files to process

Processing: Complete_French_All_in_One.pdf


Ignoring wrong pointing object 17 0 (offset 0)
Ignoring wrong pointing object 33 0 (offset 0)
Ignoring wrong pointing object 47 0 (offset 0)
Ignoring wrong pointing object 50 0 (offset 0)
Ignoring wrong pointing object 52 0 (offset 0)
Ignoring wrong pointing object 126 0 (offset 0)
Ignoring wrong pointing object 164 0 (offset 0)
Ignoring wrong pointing object 181 0 (offset 0)
Ignoring wrong pointing object 1124 0 (offset 0)
Ignoring wrong pointing object 1269 0 (offset 0)
Ignoring wrong pointing object 1536 0 (offset 0)
Ignoring wrong pointing object 1775 0 (offset 0)
Ignoring wrong pointing object 1812 0 (offset 0)
Ignoring wrong pointing object 2573 0 (offset 0)
Ignoring wrong pointing object 2592 0 (offset 0)
Ignoring wrong pointing object 2701 0 (offset 0)


 ✔ loaded 657 pages

Processing: French_Teach_Yourself _Adams_Wilson.pdf
 ✔ loaded 244 pages

Total Documents Loaded: 901


In [3]:
all_pdf_documents

[Document(metadata={'producer': 'iOS Version 15.6.1 (Build 19G82) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20220828055256Z00'00'", 'moddate': "D:20220828055256Z00'00'", 'source': '..\\data\\pdf\\Complete_French_All_in_One.pdf', 'total_pages': 657, 'page': 0, 'page_label': '1', 'source_file': 'Complete_French_All_in_One.pdf', 'file_type': 'pdf'}, page_content=''),
 Document(metadata={'producer': 'iOS Version 15.6.1 (Build 19G82) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20220828055256Z00'00'", 'moddate': "D:20220828055256Z00'00'", 'source': '..\\data\\pdf\\Complete_French_All_in_One.pdf', 'total_pages': 657, 'page': 1, 'page_label': '2', 'source_file': 'Complete_French_All_in_One.pdf', 'file_type': 'pdf'}, page_content='PRACTICE\nMAKES\nPERFECT\nComplete  \nFrench  \nAll-in-One\n®\ni-x_1-284_Heminway.indd   1 5/29/18   9:09 AM'),
 Document(metadata={'producer': 'iOS Version 15.6.1 (Build 19G82) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D

In [4]:
### Text splitting get into chunks

def split_documents(documents, chunk_size = 400, chunk_overlap = 100):
    """Split documents into smaller chunks for better RAG Performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap,
        length_function = len,
        separators = ["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show Example Of a chunk

    if split_docs:
        print(f"\nExample Chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    return split_docs 

In [5]:
chunks = split_documents(all_pdf_documents)
chunks

Split 901 documents into 4375 chunks

Example Chunk:
Content: PRACTICE
MAKES
PERFECT
Complete  
French  
All-in-One
®
i-x_1-284_Heminway.indd   1 5/29/18   9:09 AM...
Metadata: {'producer': 'iOS Version 15.6.1 (Build 19G82) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20220828055256Z00'00'", 'moddate': "D:20220828055256Z00'00'", 'source': '..\\data\\pdf\\Complete_French_All_in_One.pdf', 'total_pages': 657, 'page': 1, 'page_label': '2', 'source_file': 'Complete_French_All_in_One.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'iOS Version 15.6.1 (Build 19G82) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20220828055256Z00'00'", 'moddate': "D:20220828055256Z00'00'", 'source': '..\\data\\pdf\\Complete_French_All_in_One.pdf', 'total_pages': 657, 'page': 1, 'page_label': '2', 'source_file': 'Complete_French_All_in_One.pdf', 'file_type': 'pdf'}, page_content='PRACTICE\nMAKES\nPERFECT\nComplete  \nFrench  \nAll-in-One\n®\ni-x_1-284_Heminway.indd   1 5/29/18   9:09 AM'),
 Document(metadata={'producer': 'iOS Version 15.6.1 (Build 19G82) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20220828055256Z00'00'", 'moddate': "D:20220828055256Z00'00'", 'source': '..\\data\\pdf\\Complete_French_All_in_One.pdf', 'total_pages': 657, 'page': 2, 'page_label': '3', 'source_file': 'Complete_French_All_in_One.pdf', 'file_type': 'pdf'}, page_content='Premium Second Edition \nAnnie Heminway, Editor \nComplete  \nFrench  \nAll-in-One\ni-x_1-284_Heminway.indd   2 5/29/18   9:09 A

### Embedding and VectorDB Store

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:
    """Handles Document Embedding Generation Using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: HuggingFace model name for sentence embedding        
        """
        self.model_name = model_name
        self.model = None
        self._load_model()
    def _load_model(self):
        """Load the SentenceTransformer model"""

        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model Loaded Successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error Loading Model {self.model_name}: {e}")
            raise
    
    def generate_embedding(self, texts: List[str]) -> np.ndarray:
        """
        Generate Embedding for a list of texts

        Args:
            texts: List of text string to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model Not Loaded")
        print(f"Generating Embeddings For {len(texts)} texts....")
        embeddings = self.model.encode(texts, show_progress_bar = True)
        print(f"Generate Embedding With Shape: {embeddings.shape}")
        return embeddings
    
    #def get_embedding_dimension(self) -> int:
       ## if not self.model:
        #    raise ValueError("Model Not Loading")
        #return self.model.get_sentence_embedding_dimension()

## INitialize the embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1460.41it/s]


Model Loaded Successfully. Embedding dimension: 384


C:\Users\HP\AppData\Local\Temp\ipykernel_6928\3768047125.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model Loaded Successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


### Vector Store

In [8]:
class VectorStore:
    """Manages document embeddings in a chromaDB vector Store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vectore store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store() 
    def _initialize_store(self):
        """Initialize ChromaDB client and collections"""
        try:
            # Create persistant chromaDB Client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path = self.persist_directory)

            # Get or create collection 
            self.collection = self.client.get_or_create_collection(
                name = self.collection_name,
                metadata={"description": "PDF Documents Embedding For RAG"}
            ) 
            print(f"Vector store initialized. Collection:{self.collection_name}")
            print(f"Existing documens in collections: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store

        Args:
            documents: List of langchain documents
            embeddings: Coresponding embedding for the documents
        """ 
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        print(f"Adding {len(documents)} documents to the vector store")

        # Prepare data for chromadb

        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate Unique id
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare Metadata
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collections
        try:
            self.collection.add(
                ids = ids,
                embeddings = embeddings_list,
                metadatas = metadatas,
                documents = documents_text
            )
            print(f"Successfully added {len(documents)} Documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vector_store = VectorStore()
vector_store

Vector store initialized. Collection:pdf_documents
Existing documens in collections: 0


In [9]:
chunks

[Document(metadata={'producer': 'iOS Version 15.6.1 (Build 19G82) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20220828055256Z00'00'", 'moddate': "D:20220828055256Z00'00'", 'source': '..\\data\\pdf\\Complete_French_All_in_One.pdf', 'total_pages': 657, 'page': 1, 'page_label': '2', 'source_file': 'Complete_French_All_in_One.pdf', 'file_type': 'pdf'}, page_content='PRACTICE\nMAKES\nPERFECT\nComplete  \nFrench  \nAll-in-One\n®\ni-x_1-284_Heminway.indd   1 5/29/18   9:09 AM'),
 Document(metadata={'producer': 'iOS Version 15.6.1 (Build 19G82) Quartz PDFContext', 'creator': 'PyPDF', 'creationdate': "D:20220828055256Z00'00'", 'moddate': "D:20220828055256Z00'00'", 'source': '..\\data\\pdf\\Complete_French_All_in_One.pdf', 'total_pages': 657, 'page': 2, 'page_label': '3', 'source_file': 'Complete_French_All_in_One.pdf', 'file_type': 'pdf'}, page_content='Premium Second Edition \nAnnie Heminway, Editor \nComplete  \nFrench  \nAll-in-One\ni-x_1-284_Heminway.indd   2 5/29/18   9:09 A

In [10]:
### Convert the txt to embeddings

texts = [doc.page_content for doc in chunks]

## Generate the embedding

embeddings = embedding_manager.generate_embedding(texts)

## Store in the vector database

vector_store.add_documents(chunks, embeddings)


Generating Embeddings For 4375 texts....


Batches:   0%|          | 0/137 [00:00<?, ?it/s]

Batches: 100%|██████████| 137/137 [07:33<00:00,  3.31s/it]


Generate Embedding With Shape: (4375, 384)
Adding 4375 documents to the vector store
Successfully added 4375 Documents to vector store
Total documents in collection: 4375


### Retriever Pipeline From Vector Store

In [36]:
from turtle import distance


class RAGRetriever:
    """Handles query based retrival from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize The Retriever

        Args:
            vector_store: Vector Store containing document embeddings
            embedding_manager: Manager for generating query embedding
        
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
    

    def retrieve(self, query: str, top_k: int = 2, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve Relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum Similarity Score Threshold

        Return:
              List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top k: {top_k}, score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embedding([query])[0]

        # search in vector store

        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # convert distance to similaity store (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append(
                            {
                                'id': doc_id,
                                'content': document,
                                'metadata': metadata,
                                'similarity_score': similarity_score,
                                'distance': distance,
                                'rank': i + 1
                            }
                        )
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            return retrieved_docs
        except Exception as e:
            print(f"Error during Retrieval: {e}")
            return[]

rag_retriever = RAGRetriever(vector_store, embedding_manager)


In [37]:
rag_retriever

In [38]:
rag_retriever.retrieve("How to say hello in French")

Retrieving documents for query: 'How to say hello in French'
Top k: 2, score threshold: 0.0
Generating Embeddings For 1 texts....


Batches: 100%|██████████| 1/1 [00:00<00:00, 14.78it/s]

Generate Embedding With Shape: (1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_ec427c35_2895',
  'content': 'French in conversation: Meeting people 479\n3.     Chloé est américaine.\n4.     Chris est l’ami de Didier.\n5.     Chloé traduit des emails en français.\nReview the following explanations of some interesting phrases found in the previous dialogue. \nMake them your own.\nBonjour\nTo say hello, the words bonjour (literally, good day), bonsoir (literally, good evening), or salut (hi)',
  'metadata': {'moddate': "D:20220828055256Z00'00'",
   'page': 489,
   'file_type': 'pdf',
   'content_length': 377,
   'creator': 'PyPDF',
   'total_pages': 657,
   'creationdate': "D:20220828055256Z00'00'",
   'page_label': '490',
   'producer': 'iOS Version 15.6.1 (Build 19G82) Quartz PDFContext',
   'source': '..\\data\\pdf\\Complete_French_All_in_One.pdf',
   'source_file': 'Complete_French_All_in_One.pdf',
   'doc_index': 2895},
  'similarity_score': 0.4208187460899353,
  'distance': 0.5791812539100647,
  'rank': 1},
 {'id': 'doc_0e7ef43f_2917',
  'content'

### Integration Vector DB context pipeline with LLM output

In [39]:
import requests

def inference(prompt):
    r = requests.post("http://localhost:11434/api/generate", json = {
        #"model" : "deepseek-r1",
        "model" : "llama3.2",
        "prompt" : prompt,
        "stream" : False
    })

    response = r.json()
    return response


PROMPT_TEMPLATE = """
You are an experienced and friendly French language teacher.

Your task is to teach French using ONLY the information provided in the context below.

Instructions:
1. Answer the student's question using ONLY the provided context.
2. Do not use outside knowledge.
3. If the answer cannot be found in the context, say:
   "I could not find enough information in the provided lesson materials to answer this question."
4. Explain concepts clearly and simply, as if teaching a beginner.
5. When introducing French words or phrases:
   - Show the French expression.
   - Provide its English meaning.
   - If available, give an example sentence.
6. If the context contains examples, use them in your explanation.
7. Keep the response concise but educational.

Context:
{context}


Student Question:
{question}

"""

query = "How to say hello in French"


retrieved_docs = rag_retriever.retrieve(query)


context = "\n\n".join(
        doc['content'][:400] for doc in retrieved_docs[:2]
)


prompt = PROMPT_TEMPLATE.format(

        context=context,
        question=query

)

with open("prompt.txt", "w") as f:
    f.write(prompt)

response = inference(prompt)["response"]
print(response)

with open("response.txt", "w") as f:
    f.write(response)


Retrieving documents for query: 'How to say hello in French'
Top k: 2, score threshold: 0.0
Generating Embeddings For 1 texts....


Batches: 100%|██████████| 1/1 [00:00<00:00, 15.37it/s]

Generate Embedding With Shape: (1, 384)
Retrieved 2 documents (after filtering)


Bonjour!

To say hello in French, we have three options: "bonjour" (good day), "bonsoir" (good evening), or "salut" (hi).

Let's start with "bonjour". It's a classic greeting that you can use during the day. For example:

"Bonjour, comment ça va?"

Which means "Hello, how are you?"

The other two options are used in different situations. "Bonsoir" is used in the evening or at night, while "salut" is a more casual way to say hello, especially with friends.

So, which one do you want to use? "Bonjour", "bonsoir", or "salut"?
